In [2]:
# ============================================================
# CHECK GPU TENSORFLOW
# ============================================================

import tensorflow as tf

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices('GPU')

print("Available GPUs:", gpus)

if gpus:
    print("GPU DETECTED")
else:
    print("GPU NOT DETECTED")

TensorFlow version: 2.16.1
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU DETECTED


In [ ]:
# ============================================================
# Cell 1 - Library dan konfigurasi utama
# ============================================================

from pathlib import Path
import os
import json
import random
import pickle
import itertools
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

warnings.filterwarnings("ignore")

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x


# ============================================================
# SEED CONFIG
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# DATA CONFIG
# ============================================================

N_TARGET = 369
WINDOW_SIZE = 20
STRIDE = 2

N_FOLDER_NAME = f"N{N_TARGET:04d}"

SAMPLED_ROOT = Path("/media/spell/Spell-lab/Lidar/E.Sampled Dataset") / N_FOLDER_NAME
TRAINING_OUT_ROOT = Path("/media/spell/Spell-lab/Lidar/F.Training Results")

TRIAL_NAME = f"DGCNN_GRU_N{N_TARGET:04d}_T{WINDOW_SIZE:03d}_S{STRIDE:03d}"
TRIAL_ROOT = TRAINING_OUT_ROOT / TRIAL_NAME


# ============================================================
# OUTPUT FOLDERS
# ============================================================

MODELS_DIR = TRIAL_ROOT / "models"
SCALERS_DIR = TRIAL_ROOT / "scalers"
CONFIGS_DIR = TRIAL_ROOT / "configs"
HPO_DIR = TRIAL_ROOT / "hpo"
FINAL_DIR = TRIAL_ROOT / "final"
FIGURES_DIR = TRIAL_ROOT / "figures"
PREDICTIONS_DIR = TRIAL_ROOT / "predictions"
REPORTS_DIR = TRIAL_ROOT / "reports"

for d in [
    TRIAL_ROOT, MODELS_DIR, SCALERS_DIR, CONFIGS_DIR,
    HPO_DIR, FINAL_DIR, FIGURES_DIR, PREDICTIONS_DIR, REPORTS_DIR
]:
    d.mkdir(parents=True, exist_ok=True)


# ============================================================
# DATASET CONFIG
# ============================================================

ACTIVITIES = ["Bungkuk", "Duduk", "Jongkok", "Jatuh"]

DEV_SUBJECTS = [
    "Adelia", "Afi", "Aswangga", "Bustan",
    "Dilia", "Eldivo", "Fathir", "Lina",
    "Manda", "Miftah", "Teguh", "Tsamara",
]

TEST_SUBJECTS = ["Kanaya", "Naila", "Nana", "Rega", "Zaira"]

TEST_ROOM_FOR_MODEL_EVAL = "Controlled Room"

FILE_IDS = list(range(1, 10))

FEATURE_COLUMNS = ["X_corr", "Y_corr", "Z_level"]
REQUIRED_COLUMNS = [
    "frame_id", "Timestamp", "X", "Y", "Z",
    "X_corr", "Y_corr", "Z_corr", "Z_level", "Reflectivity"
]


# ============================================================
# LABEL CONFIG
# ============================================================

LABEL_MAP = {
    "Bungkuk": 0,
    "Duduk": 0,
    "Jongkok": 0,
    "Jatuh": 1,
}

CLASS_NAMES = ["Non-Fall", "Fall"]


# ============================================================
# TRAINING CONFIG
# ============================================================

BATCH_SIZE = 4

CV_SPLITS = 4
RANDOM_SEARCH_TRIALS = 10

MAX_EPOCHS_HPO = 50
EARLY_STOP_PATIENCE_HPO = 5

MAX_EPOCHS_FINAL = 100
EARLY_STOP_PATIENCE_FINAL = 15

VERBOSE = 1

FINAL_VAL_SPLITS = 6  # 12 subjects -> approx 10 train, 2 val


# ============================================================
# HPO CONFIG
# ============================================================

HPO_SPACE = {
    "knn_k": [10, 15, 20],
    "learning_rate": [5e-4, 1e-4, 5e-5],
    "l2_weight": [1e-5, 1e-4],
    "dropout_temporal": [0.2, 0.3],
    "dropout_classifier": [0.3, 0.4],
}


# ============================================================
# BEHAVIOR CONFIG
# ============================================================

RUN_HPO = True
USE_EXISTING_BEST_CONFIG_IF_AVAILABLE = True
REQUIRE_CONSECUTIVE_FRAMES = True

BEST_MODEL_PATH = MODELS_DIR / "best_model.keras"
SCALER_PATH = SCALERS_DIR / "scaler.pkl"


print("===== TRAINING CONFIG =====")
print(f"Sampled root : {SAMPLED_ROOT}")
print(f"Trial root   : {TRIAL_ROOT}")
print(f"N_TARGET     : {N_TARGET}")
print(f"WINDOW_SIZE  : {WINDOW_SIZE}")
print(f"STRIDE       : {STRIDE}")
print(f"BATCH_SIZE   : {BATCH_SIZE}")
print(f"Framework    : TensorFlow {tf.__version__}")

===== TRAINING CONFIG =====
Sampled root : /media/spell/Spell-lab/Lidar/E.Sampled Dataset/N0369
Trial root   : /media/spell/Spell-lab/Lidar/F.Training Results/DGCNN_GRU_N0369_T020_S002
N_TARGET     : 369
WINDOW_SIZE  : 20
STRIDE       : 2
BATCH_SIZE   : 4
Framework    : TensorFlow 2.16.1


In [4]:
# ============================================================
# Cell 2 - Simpan konfigurasi training
# ============================================================

training_config = {
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "seed": SEED,
    "sampled_root": str(SAMPLED_ROOT),
    "training_out_root": str(TRAINING_OUT_ROOT),
    "trial_name": TRIAL_NAME,
    "trial_root": str(TRIAL_ROOT),
    "n_target": N_TARGET,
    "window_size": WINDOW_SIZE,
    "stride": STRIDE,
    "feature_columns": FEATURE_COLUMNS,
    "label_map": LABEL_MAP,
    "class_names": CLASS_NAMES,
    "batch_size": BATCH_SIZE,
    "cv_splits": CV_SPLITS,
    "random_search_trials": RANDOM_SEARCH_TRIALS,
    "max_epochs_hpo": MAX_EPOCHS_HPO,
    "early_stop_patience_hpo": EARLY_STOP_PATIENCE_HPO,
    "max_epochs_final": MAX_EPOCHS_FINAL,
    "early_stop_patience_final": EARLY_STOP_PATIENCE_FINAL,
    "hpo_space": HPO_SPACE,
    "test_room_for_model_eval": TEST_ROOM_FOR_MODEL_EVAL,
}

training_config_path = CONFIGS_DIR / "training_config.json"

with open(training_config_path, "w") as f:
    json.dump(training_config, f, indent=4)

print(f"Training config saved to: {training_config_path}")

Training config saved to: /media/spell/Spell-lab/Lidar/F.Training Results/DGCNN_GRU_N0369_T020_S002/configs/training_config.json


In [5]:
# ============================================================
# Cell 3 - Helper manifest dataset
# ============================================================

def build_development_manifest():
    records = []

    for activity in ACTIVITIES:
        for subject in DEV_SUBJECTS:
            for file_id in FILE_IDS:
                file_name = f"{file_id}.csv"

                path = (
                    SAMPLED_ROOT
                    / "Dataset Development"
                    / activity
                    / subject
                    / file_name
                )

                records.append({
                    "dataset": "development",
                    "room": "",
                    "activity": activity,
                    "subject": subject,
                    "file_id": file_id,
                    "file_name": file_name,
                    "path": path,
                    "label": LABEL_MAP[activity],
                    "exists": path.exists(),
                })

    return pd.DataFrame(records)


def build_controlled_test_manifest():
    records = []

    for activity in ACTIVITIES:
        for subject in TEST_SUBJECTS:
            for file_id in FILE_IDS:
                file_name = f"{file_id}.csv"

                path = (
                    SAMPLED_ROOT
                    / "Dataset Testing"
                    / TEST_ROOM_FOR_MODEL_EVAL
                    / activity
                    / subject
                    / file_name
                )

                records.append({
                    "dataset": "testing",
                    "room": TEST_ROOM_FOR_MODEL_EVAL,
                    "activity": activity,
                    "subject": subject,
                    "file_id": file_id,
                    "file_name": file_name,
                    "path": path,
                    "label": LABEL_MAP[activity],
                    "exists": path.exists(),
                })

    return pd.DataFrame(records)


dev_manifest_df = build_development_manifest()
test_manifest_df = build_controlled_test_manifest()

dev_manifest_save = dev_manifest_df.copy()
test_manifest_save = test_manifest_df.copy()

dev_manifest_save["path"] = dev_manifest_save["path"].astype(str)
test_manifest_save["path"] = test_manifest_save["path"].astype(str)

dev_manifest_save.to_csv(CONFIGS_DIR / "development_manifest.csv", index=False)
test_manifest_save.to_csv(CONFIGS_DIR / "controlled_test_manifest.csv", index=False)

print("===== MANIFEST SUMMARY =====")
print(f"Development expected files : {len(dev_manifest_df)}")
print(f"Development existing files : {dev_manifest_df['exists'].sum()}")
print(f"Development missing files  : {(~dev_manifest_df['exists']).sum()}")
print()
print(f"Testing expected files     : {len(test_manifest_df)}")
print(f"Testing existing files     : {test_manifest_df['exists'].sum()}")
print(f"Testing missing files      : {(~test_manifest_df['exists']).sum()}")

if dev_manifest_df["exists"].sum() == 0:
    raise FileNotFoundError("Tidak ada file development ditemukan.")

if test_manifest_df["exists"].sum() == 0:
    raise FileNotFoundError("Tidak ada file controlled testing ditemukan.")

===== MANIFEST SUMMARY =====
Development expected files : 432
Development existing files : 432
Development missing files  : 0

Testing expected files     : 180
Testing existing files     : 180
Testing missing files      : 0


In [6]:
# ============================================================
# Cell 4 - Helper load CSV dan build window
# ============================================================

def load_sampled_csv(path):
    df = pd.read_csv(path)

    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in {path}: {missing_cols}")

    df = df[REQUIRED_COLUMNS].copy()

    for col in ["frame_id"] + FEATURE_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["frame_id"] + FEATURE_COLUMNS).copy()
    df["frame_id"] = df["frame_id"].astype(int)

    return df


def build_windows_from_file(row):
    path = Path(row["path"])

    if not path.exists():
        return [], []

    df = load_sampled_csv(path)

    frame_ids = sorted(df["frame_id"].unique().tolist())
    frame_dict = {}

    for fid, g in df.groupby("frame_id"):
        if len(g) != N_TARGET:
            continue

        arr = g[FEATURE_COLUMNS].to_numpy(dtype=np.float32)

        if arr.shape != (N_TARGET, len(FEATURE_COLUMNS)):
            continue

        frame_dict[int(fid)] = arr

    usable_frame_ids = sorted(frame_dict.keys())

    windows = []
    metas = []

    for start_idx in range(0, len(usable_frame_ids) - WINDOW_SIZE + 1, STRIDE):
        win_frame_ids = usable_frame_ids[start_idx:start_idx + WINDOW_SIZE]

        if REQUIRE_CONSECUTIVE_FRAMES:
            diffs = np.diff(win_frame_ids)
            if not np.all(diffs == 1):
                continue

        window_arr = np.stack(
            [frame_dict[fid] for fid in win_frame_ids],
            axis=0
        )

        windows.append(window_arr)

        metas.append({
            "dataset": row["dataset"],
            "room": row["room"],
            "activity": row["activity"],
            "subject": row["subject"],
            "file_id": int(row["file_id"]),
            "label": int(row["label"]),
            "start_frame": int(win_frame_ids[0]),
            "end_frame": int(win_frame_ids[-1]),
            "source_path": str(path),
        })

    return windows, metas

In [7]:
# ============================================================
# Cell 5 - Build semua windows development dan test
# ============================================================

def build_dataset_windows(manifest_df, desc="Building windows"):
    all_windows = []
    all_metas = []

    existing_manifest = manifest_df[manifest_df["exists"]].copy()

    for _, row in tqdm(existing_manifest.iterrows(), total=len(existing_manifest), desc=desc):
        windows, metas = build_windows_from_file(row)

        if len(windows) > 0:
            all_windows.extend(windows)
            all_metas.extend(metas)

    if len(all_windows) == 0:
        raise ValueError(f"Tidak ada window terbentuk untuk: {desc}")

    X = np.stack(all_windows, axis=0).astype(np.float32)
    meta_df = pd.DataFrame(all_metas)
    y = meta_df["label"].to_numpy(dtype=np.int32)
    groups = meta_df["subject"].to_numpy()

    return X, y, groups, meta_df


X_dev, y_dev, groups_dev, meta_dev_df = build_dataset_windows(
    dev_manifest_df,
    desc="Building development windows"
)

X_test, y_test, groups_test, meta_test_df = build_dataset_windows(
    test_manifest_df,
    desc="Building controlled test windows"
)

meta_dev_df.to_csv(CONFIGS_DIR / "development_windows_metadata.csv", index=False)
meta_test_df.to_csv(CONFIGS_DIR / "controlled_test_windows_metadata.csv", index=False)

print("===== WINDOW DATASET SUMMARY =====")
print(f"X_dev shape  : {X_dev.shape}")
print(f"y_dev shape  : {y_dev.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_test shape : {y_test.shape}")
print()
print("Development class distribution:")
display(pd.Series(y_dev).value_counts().sort_index().rename(index={0: "Non-Fall", 1: "Fall"}))
print()
print("Test class distribution:")
display(pd.Series(y_test).value_counts().sort_index().rename(index={0: "Non-Fall", 1: "Fall"}))

Building development windows:   0%|          | 0/432 [00:00<?, ?it/s]

Building controlled test windows:   0%|          | 0/180 [00:00<?, ?it/s]

===== WINDOW DATASET SUMMARY =====
X_dev shape  : (9253, 20, 369, 3)
y_dev shape  : (9253,)
X_test shape : (3767, 20, 369, 3)
y_test shape : (3767,)

Development class distribution:


Non-Fall    6792
Fall        2461
Name: count, dtype: int64


Test class distribution:


Non-Fall    2753
Fall        1014
Name: count, dtype: int64

In [8]:
# ============================================================
# Cell 6 - Helper scaling bebas leakage
# ============================================================

def fit_scaler_on_train(X_train):
    scaler = StandardScaler()

    flat_train = X_train.reshape(-1, X_train.shape[-1])
    scaler.fit(flat_train)

    return scaler


def transform_with_scaler(X, scaler):
    original_shape = X.shape
    flat = X.reshape(-1, X.shape[-1])
    flat_scaled = scaler.transform(flat)
    return flat_scaled.reshape(original_shape).astype(np.float32)

In [9]:
# ============================================================
# Cell 7 - Helper class weight dan tf.data
# ============================================================

def compute_class_weight_dict(y_train):
    classes = np.array([0, 1], dtype=np.int32)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train,
    )

    return {int(c): float(w) for c, w in zip(classes, weights)}


def make_tf_dataset(X, y, batch_size=BATCH_SIZE, shuffle=False, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices((X.astype(np.float32), y.astype(np.float32)))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(y), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [10]:
# ============================================================
# Cell 8 - Layer EdgeConv DGCNN
# ============================================================

class EdgeConvBlock(layers.Layer):
    def __init__(self, out_channels, k=20, l2_weight=1e-5, name=None):
        super().__init__(name=name)
        self.out_channels = out_channels
        self.k = k
        self.l2_weight = l2_weight

        self.conv = layers.Conv2D(
            filters=out_channels,
            kernel_size=(1, 1),
            padding="valid",
            use_bias=False,
            kernel_regularizer=regularizers.l2(l2_weight),
        )
        self.bn = layers.BatchNormalization()
        self.act = layers.Activation("relu")

    def call(self, x, training=False):
        # x shape: [B, N, C]
        batch_size = tf.shape(x)[0]
        num_points = tf.shape(x)[1]

        inner = -2.0 * tf.matmul(x, x, transpose_b=True)
        xx = tf.reduce_sum(tf.square(x), axis=-1, keepdims=True)
        pairwise_dist = -xx - inner - tf.transpose(xx, perm=[0, 2, 1])

        k = tf.minimum(self.k, num_points)

        _, idx = tf.nn.top_k(pairwise_dist, k=k)

        neighbors = tf.gather(x, idx, batch_dims=1)

        x_central = tf.expand_dims(x, axis=2)
        x_central = tf.tile(x_central, [1, 1, k, 1])

        edge_feature = tf.concat([x_central, neighbors - x_central], axis=-1)

        out = self.conv(edge_feature)
        out = self.bn(out, training=training)
        out = self.act(out)

        out = tf.reduce_max(out, axis=2)

        return out

In [11]:
# ============================================================
# Cell 9 - Build model DGCNN-GRU
# ============================================================

def build_dgcnn_gru_model(config):
    knn_k = int(config["knn_k"])
    l2_weight = float(config["l2_weight"])
    dropout_temporal = float(config["dropout_temporal"])
    dropout_classifier = float(config["dropout_classifier"])
    learning_rate = float(config["learning_rate"])

    input_layer = layers.Input(
        shape=(WINDOW_SIZE, N_TARGET, len(FEATURE_COLUMNS)),
        name="input_sequence"
    )

    # [B, T, N, C] -> [B*T, N, C]
    x = layers.Lambda(
        lambda z: tf.reshape(z, (-1, N_TARGET, len(FEATURE_COLUMNS))),
        name="merge_batch_time"
    )(input_layer)

    # DGCNN spatial encoder
    x1 = EdgeConvBlock(64, k=knn_k, l2_weight=l2_weight, name="edgeconv_64_a")(x)
    x2 = EdgeConvBlock(64, k=knn_k, l2_weight=l2_weight, name="edgeconv_64_b")(x1)
    x3 = EdgeConvBlock(128, k=knn_k, l2_weight=l2_weight, name="edgeconv_128")(x2)
    x4 = EdgeConvBlock(256, k=knn_k, l2_weight=l2_weight, name="edgeconv_256")(x3)

    global_max = layers.GlobalMaxPooling1D(name="global_max_pool")(x4)
    global_avg = layers.GlobalAveragePooling1D(name="global_avg_pool")(x4)

    frame_feat = layers.Concatenate(name="max_avg_concat")([global_max, global_avg])

    frame_feat = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=regularizers.l2(l2_weight),
        name="frame_projection_256"
    )(frame_feat)

    frame_feat = layers.BatchNormalization(name="frame_projection_bn")(frame_feat)

    # [B*T, D] -> [B, T, D]
    seq_feat = layers.Lambda(
        lambda z: tf.reshape(z, (-1, WINDOW_SIZE, 256)),
        name="restore_time_dimension"
    )(frame_feat)

    # GRU temporal encoder
    seq_feat = layers.GRU(
        128,
        return_sequences=True,
        kernel_regularizer=regularizers.l2(l2_weight),
        name="gru_128"
    )(seq_feat)

    seq_feat = layers.GRU(
        64,
        return_sequences=False,
        kernel_regularizer=regularizers.l2(l2_weight),
        name="gru_64"
    )(seq_feat)

    seq_feat = layers.Dropout(dropout_temporal, name="temporal_dropout")(seq_feat)

    # Classifier
    out = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(l2_weight),
        name="classifier_dense_128"
    )(seq_feat)

    out = layers.BatchNormalization(name="classifier_bn")(out)
    out = layers.Dropout(dropout_classifier, name="classifier_dropout")(out)

    output = layers.Dense(1, activation="sigmoid", name="fall_probability")(out)

    model = models.Model(inputs=input_layer, outputs=output, name="DGCNN_GRU_FallDetection")

    try:
        optimizer = tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=l2_weight,
        )
    except Exception:
        optimizer = tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
        )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model

In [12]:
# ============================================================
# Cell 10 - Helper metric evaluasi
# ============================================================

def evaluate_binary_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_binary": precision_score(y_true, y_pred, zero_division=0),
        "recall_binary": recall_score(y_true, y_pred, zero_division=0),
        "f1_binary": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "fall_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "fall_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

    return metrics, y_pred


def get_callbacks(model_path, patience):
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=7,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(model_path),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=False,
            mode="min",
            verbose=1,
        ),
    ]

    return callbacks

In [13]:
# ============================================================
# Cell 11 - Random Search HPO configs
# ============================================================

def generate_random_hpo_configs(hpo_space, n_trials, seed=SEED):
    keys = list(hpo_space.keys())
    all_combinations = list(itertools.product(*[hpo_space[k] for k in keys]))

    rng = np.random.default_rng(seed)
    rng.shuffle(all_combinations)

    selected = all_combinations[:min(n_trials, len(all_combinations))]

    configs = []
    for values in selected:
        configs.append({k: v for k, v in zip(keys, values)})

    return configs


hpo_configs = generate_random_hpo_configs(
    HPO_SPACE,
    RANDOM_SEARCH_TRIALS,
    seed=SEED
)

print("===== HPO CONFIGS =====")
for i, cfg in enumerate(hpo_configs, 1):
    print(f"Trial {i}: {cfg}")

===== HPO CONFIGS =====
Trial 1: {'knn_k': 20, 'learning_rate': 0.0005, 'l2_weight': 1e-05, 'dropout_temporal': 0.2, 'dropout_classifier': 0.3}
Trial 2: {'knn_k': 15, 'learning_rate': 0.0005, 'l2_weight': 0.0001, 'dropout_temporal': 0.2, 'dropout_classifier': 0.3}
Trial 3: {'knn_k': 10, 'learning_rate': 5e-05, 'l2_weight': 1e-05, 'dropout_temporal': 0.3, 'dropout_classifier': 0.3}
Trial 4: {'knn_k': 15, 'learning_rate': 0.0005, 'l2_weight': 1e-05, 'dropout_temporal': 0.2, 'dropout_classifier': 0.4}
Trial 5: {'knn_k': 15, 'learning_rate': 0.0001, 'l2_weight': 0.0001, 'dropout_temporal': 0.3, 'dropout_classifier': 0.4}
Trial 6: {'knn_k': 15, 'learning_rate': 0.0001, 'l2_weight': 0.0001, 'dropout_temporal': 0.2, 'dropout_classifier': 0.4}
Trial 7: {'knn_k': 20, 'learning_rate': 0.0001, 'l2_weight': 0.0001, 'dropout_temporal': 0.2, 'dropout_classifier': 0.3}
Trial 8: {'knn_k': 20, 'learning_rate': 0.0005, 'l2_weight': 1e-05, 'dropout_temporal': 0.3, 'dropout_classifier': 0.3}
Trial 9: {'kn

In [14]:
# ============================================================
# Cell 12 - Run HPO dengan StratifiedGroupKFold
# ============================================================

best_config_path = CONFIGS_DIR / "best_hpo_config.json"

all_hpo_results = []
all_fold_results = []

if USE_EXISTING_BEST_CONFIG_IF_AVAILABLE and best_config_path.exists():
    print(f"Existing best config found: {best_config_path}")
    with open(best_config_path, "r") as f:
        best_hpo_config = json.load(f)
    RUN_HPO_EFFECTIVE = False
else:
    RUN_HPO_EFFECTIVE = RUN_HPO


if RUN_HPO_EFFECTIVE:
    cv = StratifiedGroupKFold(
        n_splits=CV_SPLITS,
        shuffle=True,
        random_state=SEED,
    )

    for trial_idx, config in enumerate(hpo_configs, start=1):
        print("\n" + "=" * 70)
        print(f"HPO TRIAL {trial_idx}/{len(hpo_configs)}")
        print(config)
        print("=" * 70)

        fold_metrics_list = []

        for fold_idx, (train_idx, val_idx) in enumerate(
            cv.split(X_dev, y_dev, groups=groups_dev),
            start=1
        ):
            print(f"\n--- Trial {trial_idx} | Fold {fold_idx}/{CV_SPLITS} ---")

            tf.keras.backend.clear_session()

            X_train_raw, X_val_raw = X_dev[train_idx], X_dev[val_idx]
            y_train, y_val = y_dev[train_idx], y_dev[val_idx]

            scaler = fit_scaler_on_train(X_train_raw)
            X_train = transform_with_scaler(X_train_raw, scaler)
            X_val = transform_with_scaler(X_val_raw, scaler)

            train_ds = make_tf_dataset(X_train, y_train, shuffle=True, seed=SEED + fold_idx)
            val_ds = make_tf_dataset(X_val, y_val, shuffle=False)

            class_weight_dict = compute_class_weight_dict(y_train)

            fold_model_path = HPO_DIR / f"trial_{trial_idx:02d}_fold_{fold_idx:02d}.keras"

            model = build_dgcnn_gru_model(config)

            callbacks = get_callbacks(
                model_path=fold_model_path,
                patience=EARLY_STOP_PATIENCE_HPO,
            )

            history = model.fit(
                train_ds,
                validation_data=val_ds,
                epochs=MAX_EPOCHS_HPO,
                class_weight=class_weight_dict,
                callbacks=callbacks,
                verbose=VERBOSE,
            )

            best_model = tf.keras.models.load_model(
                fold_model_path,
                custom_objects={"EdgeConvBlock": EdgeConvBlock}
            )

            y_val_prob = best_model.predict(val_ds, verbose=0).ravel()
            fold_metrics, y_val_pred = evaluate_binary_predictions(y_val, y_val_prob)

            fold_metrics_record = {
                "trial_idx": trial_idx,
                "fold_idx": fold_idx,
                **config,
                **fold_metrics,
                "train_subjects": ",".join(sorted(set(groups_dev[train_idx]))),
                "val_subjects": ",".join(sorted(set(groups_dev[val_idx]))),
                "n_train": len(train_idx),
                "n_val": len(val_idx),
            }

            all_fold_results.append(fold_metrics_record)
            fold_metrics_list.append(fold_metrics)

            print("Fold metrics:")
            print(fold_metrics)

        avg_metrics = {
            key: float(np.mean([m[key] for m in fold_metrics_list]))
            for key in fold_metrics_list[0].keys()
        }

        trial_result = {
            "trial_idx": trial_idx,
            **config,
            **{f"mean_{k}": v for k, v in avg_metrics.items()},
        }

        all_hpo_results.append(trial_result)

        print("\nTrial average metrics:")
        print(trial_result)

    hpo_results_df = pd.DataFrame(all_hpo_results)
    fold_results_df = pd.DataFrame(all_fold_results)

    hpo_results_path = HPO_DIR / "all_hpo_trial_results.csv"
    fold_results_path = HPO_DIR / "cv_fold_metrics.csv"

    hpo_results_df.to_csv(hpo_results_path, index=False)
    fold_results_df.to_csv(fold_results_path, index=False)

    # Ranking: primary macro-F1, secondary fall recall, tertiary balanced accuracy
    hpo_results_df = hpo_results_df.sort_values(
        by=["mean_macro_f1", "mean_fall_recall", "mean_balanced_accuracy"],
        ascending=False,
    ).reset_index(drop=True)

    best_hpo_config = {
        "knn_k": int(hpo_results_df.loc[0, "knn_k"]),
        "learning_rate": float(hpo_results_df.loc[0, "learning_rate"]),
        "l2_weight": float(hpo_results_df.loc[0, "l2_weight"]),
        "dropout_temporal": float(hpo_results_df.loc[0, "dropout_temporal"]),
        "dropout_classifier": float(hpo_results_df.loc[0, "dropout_classifier"]),
        "selection_rule": "max mean_macro_f1, then mean_fall_recall, then mean_balanced_accuracy",
        "best_trial_idx": int(hpo_results_df.loc[0, "trial_idx"]),
        "best_mean_macro_f1": float(hpo_results_df.loc[0, "mean_macro_f1"]),
        "best_mean_fall_recall": float(hpo_results_df.loc[0, "mean_fall_recall"]),
        "best_mean_balanced_accuracy": float(hpo_results_df.loc[0, "mean_balanced_accuracy"]),
    }

    with open(best_config_path, "w") as f:
        json.dump(best_hpo_config, f, indent=4)

    print("\n===== BEST HPO CONFIG =====")
    print(best_hpo_config)

else:
    print("Skipping HPO and using existing best config:")
    print(best_hpo_config)


HPO TRIAL 1/10
{'knn_k': 20, 'learning_rate': 0.0005, 'l2_weight': 1e-05, 'dropout_temporal': 0.2, 'dropout_classifier': 0.3}

--- Trial 1 | Fold 1/4 ---


2026-05-11 08:21:26.648955: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:00:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-11 08:21:26.649149: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:00:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-11 08:21:26.649225: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:00:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-11 08:21:26.720481: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:00:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-11 08:21:26.720686: I external/local_xla/xla/stream_executor

Epoch 1/50


2026-05-11 08:21:46.648983: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 90300


1754/1754 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7111 - loss: 0.5940
Epoch 1: val_loss improved from None to 0.28220, saving model to /media/spell/Spell-lab/Lidar/F.Training Results/DGCNN_GRU_N0369_T020_S002/hpo/trial_01_fold_01.keras
1754/1754 ━━━━━━━━━━━━━━━━━━━━ 3460s 2s/step - accuracy: 0.7467 - loss: 0.5513 - val_accuracy: 0.9294 - val_loss: 0.2822 - learning_rate: 5.0000e-04
Epoch 2/50
1754/1754 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8005 - loss: 0.4692
Epoch 2: val_loss improved from 0.28220 to 0.19905, saving model to /media/spell/Spell-lab/Lidar/F.Training Results/DGCNN_GRU_N0369_T020_S002/hpo/trial_01_fold_01.keras
1754/1754 ━━━━━━━━━━━━━━━━━━━━ 3488s 2s/step - accuracy: 0.8060 - loss: 0.4608 - val_accuracy: 0.9446 - val_loss: 0.1990 - learning_rate: 5.0000e-04
Epoch 3/50
1754/1754 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8218 - loss: 0.4250
Epoch 3: val_loss did not improve from 0.19905
1754/1754 ━━━━━━━━━━━━━━━━━━━━ 3513s 2s/step - accuracy: 0.8215 - loss

ValueError: Requested the deserialization of a `Lambda` layer whose `function` is a Python lambda. This carries a potential risk of arbitrary code execution and thus it is disallowed by default. If you trust the source of the artifact, you can override this error by passing `safe_mode=False` to the loading function, or calling `keras.config.enable_unsafe_deserialization().

In [ ]:
# ============================================================
# Cell 13 - Final train/val subject-wise split
# ============================================================

final_splitter = StratifiedGroupKFold(
    n_splits=FINAL_VAL_SPLITS,
    shuffle=True,
    random_state=SEED,
)

split_candidates = list(final_splitter.split(X_dev, y_dev, groups=groups_dev))

# Pilih split pertama yang otomatis menghasilkan sekitar 10 train subject dan 2 val subject
final_train_idx, final_val_idx = split_candidates[0]

final_train_subjects = sorted(set(groups_dev[final_train_idx]))
final_val_subjects = sorted(set(groups_dev[final_val_idx]))

print("===== FINAL TRAIN/VAL SPLIT =====")
print(f"Final train subjects ({len(final_train_subjects)}): {final_train_subjects}")
print(f"Final val subjects   ({len(final_val_subjects)}): {final_val_subjects}")
print(f"Final train windows: {len(final_train_idx)}")
print(f"Final val windows  : {len(final_val_idx)}")

# Safety check final train/val subject split
assert len(final_train_subjects) == 10, final_train_subjects
assert len(final_val_subjects) == 2, final_val_subjects
assert set(final_train_subjects).isdisjoint(set(final_val_subjects))
assert len(set(final_train_subjects + final_val_subjects)) == 12

print("Final train/val subject split check: PASSED")

split_info = {
    "final_train_subjects": final_train_subjects,
    "final_val_subjects": final_val_subjects,
    "n_final_train_windows": int(len(final_train_idx)),
    "n_final_val_windows": int(len(final_val_idx)),
}

with open(CONFIGS_DIR / "final_train_val_split.json", "w") as f:
    json.dump(split_info, f, indent=4)

In [ ]:
# ============================================================
# Cell 14 - Final scaling dan dataset
# ============================================================

X_final_train_raw = X_dev[final_train_idx]
y_final_train = y_dev[final_train_idx]

X_final_val_raw = X_dev[final_val_idx]
y_final_val = y_dev[final_val_idx]

final_scaler = fit_scaler_on_train(X_final_train_raw)

X_final_train = transform_with_scaler(X_final_train_raw, final_scaler)
X_final_val = transform_with_scaler(X_final_val_raw, final_scaler)
X_test_scaled = transform_with_scaler(X_test, final_scaler)

with open(SCALER_PATH, "wb") as f:
    pickle.dump(final_scaler, f)

final_train_ds = make_tf_dataset(X_final_train, y_final_train, shuffle=True, seed=SEED)
final_val_ds = make_tf_dataset(X_final_val, y_final_val, shuffle=False)
test_ds = make_tf_dataset(X_test_scaled, y_test, shuffle=False)

final_class_weight = compute_class_weight_dict(y_final_train)

print("===== FINAL DATASET READY =====")
print(f"X_final_train : {X_final_train.shape}")
print(f"X_final_val   : {X_final_val.shape}")
print(f"X_test_scaled : {X_test_scaled.shape}")
print(f"Class weight  : {final_class_weight}")
print(f"Scaler saved  : {SCALER_PATH}")

In [ ]:
# ============================================================
# Cell 15 - Final training dengan best config
# ============================================================

tf.keras.backend.clear_session()

print("===== FINAL TRAINING CONFIG =====")
print(best_hpo_config)

final_model = build_dgcnn_gru_model(best_hpo_config)

final_model.summary()

final_callbacks = get_callbacks(
    model_path=BEST_MODEL_PATH,
    patience=EARLY_STOP_PATIENCE_FINAL,
)

history = final_model.fit(
    final_train_ds,
    validation_data=final_val_ds,
    epochs=MAX_EPOCHS_FINAL,
    class_weight=final_class_weight,
    callbacks=final_callbacks,
    verbose=VERBOSE,
)

history_df = pd.DataFrame(history.history)
history_path = FINAL_DIR / "final_training_history.csv"
history_df.to_csv(history_path, index=False)

print(f"Final training history saved to: {history_path}")
print(f"Best model checkpoint saved to: {BEST_MODEL_PATH}")

In [ ]:
# ============================================================
# Cell 16 - Plot training curves
# ============================================================

def plot_training_curves(history_df, save_path):
    plt.figure(figsize=(10, 5))
    plt.plot(history_df["loss"], label="Train Loss")
    plt.plot(history_df["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path.with_name("training_loss_curve.png"), dpi=300)
    plt.show()

    if "accuracy" in history_df.columns and "val_accuracy" in history_df.columns:
        plt.figure(figsize=(10, 5))
        plt.plot(history_df["accuracy"], label="Train Accuracy")
        plt.plot(history_df["val_accuracy"], label="Val Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title("Training and Validation Accuracy")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(save_path.with_name("training_accuracy_curve.png"), dpi=300)
        plt.show()


plot_training_curves(history_df, FIGURES_DIR / "training_curves.png")

In [ ]:
# ============================================================
# Cell 17 - Load best model dan evaluasi controlled test
# ============================================================

best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    custom_objects={"EdgeConvBlock": EdgeConvBlock}
)

test_prob = best_model.predict(test_ds, verbose=VERBOSE).ravel()

test_metrics, test_pred = evaluate_binary_predictions(y_test, test_prob)

print("===== CONTROLLED TEST METRICS =====")
for k, v in test_metrics.items():
    print(f"{k}: {v:.6f}")

metrics_summary_path = REPORTS_DIR / "metrics_summary.json"
with open(metrics_summary_path, "w") as f:
    json.dump(test_metrics, f, indent=4)

print(f"Metrics summary saved to: {metrics_summary_path}")

In [ ]:
# ============================================================
# Cell 18 - Confusion matrix dan classification report
# ============================================================

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])

print("===== CONFUSION MATRIX =====")
print(cm)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Controlled Test Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], CLASS_NAMES)
plt.yticks([0, 1], CLASS_NAMES)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.colorbar()
plt.tight_layout()

cm_path = FIGURES_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=300)
plt.show()

print(f"Confusion matrix saved to: {cm_path}")

report_dict = classification_report(
    y_test,
    test_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).transpose()
report_path = REPORTS_DIR / "classification_report.csv"
report_df.to_csv(report_path)

display(report_df)
print(f"Classification report saved to: {report_path}")

In [ ]:
# ============================================================
# Cell 19 - Simpan test predictions
# ============================================================

test_predictions_df = meta_test_df.copy()
test_predictions_df["y_true"] = y_test
test_predictions_df["y_prob_fall"] = test_prob
test_predictions_df["y_pred"] = test_pred
test_predictions_df["true_label_name"] = test_predictions_df["y_true"].map({0: "Non-Fall", 1: "Fall"})
test_predictions_df["pred_label_name"] = test_predictions_df["y_pred"].map({0: "Non-Fall", 1: "Fall"})

predictions_path = PREDICTIONS_DIR / "test_predictions.csv"
test_predictions_df.to_csv(predictions_path, index=False)

display(test_predictions_df.head())
print(f"Test predictions saved to: {predictions_path}")

In [ ]:
# ============================================================
# Cell 20 - Per-activity error analysis
# ============================================================

activity_analysis = []

for activity, g in test_predictions_df.groupby("activity"):
    y_true_act = g["y_true"].to_numpy()
    y_pred_act = g["y_pred"].to_numpy()

    activity_analysis.append({
        "activity": activity,
        "n_windows": len(g),
        "true_label": "Fall" if LABEL_MAP[activity] == 1 else "Non-Fall",
        "accuracy": accuracy_score(y_true_act, y_pred_act),
        "fall_prediction_rate": float(np.mean(y_pred_act == 1)),
        "nonfall_prediction_rate": float(np.mean(y_pred_act == 0)),
    })

activity_analysis_df = pd.DataFrame(activity_analysis)

activity_analysis_path = REPORTS_DIR / "per_activity_error_analysis.csv"
activity_analysis_df.to_csv(activity_analysis_path, index=False)

display(activity_analysis_df)
print(f"Per-activity analysis saved to: {activity_analysis_path}")

In [ ]:
# ============================================================
# Cell 21 - Final artifact check
# ============================================================

print("===== FINAL ARTIFACT CHECK =====")

artifact_paths = {
    "best_model": BEST_MODEL_PATH,
    "scaler": SCALER_PATH,
    "training_config": CONFIGS_DIR / "training_config.json",
    "best_hpo_config": CONFIGS_DIR / "best_hpo_config.json",
    "history": FINAL_DIR / "final_training_history.csv",
    "metrics": REPORTS_DIR / "metrics_summary.json",
    "classification_report": REPORTS_DIR / "classification_report.csv",
    "test_predictions": PREDICTIONS_DIR / "test_predictions.csv",
    "confusion_matrix": FIGURES_DIR / "confusion_matrix.png",
}

for name, path in artifact_paths.items():
    print(f"{name:25s}: {path.exists()} | {path}")

print("\nTrial root:")
print(TRIAL_ROOT)